# Model registry: which penguin model is live?

## Learning objectives

- Register model versions with GTO, which records them in Git tags.
- Compare validation accuracy and promote the better version.
- Push every model version with DVC and restore it into an empty folder.

## The problem

- The workshop's exercises use one project, [a study of Palmer penguins](../../project.md), and add a capability in each exercise.
- The analysis now predicts sex from bill length, bill depth, flipper length, body mass and species.
- `project` has a trained model, but no registered versions and no version marked as live.

## What you'll do

1. Register the first model, trained with `C: 0.01`.
2. Train with `C: 1`, commit and register the second model.
3. Compare their validation accuracy and assign the better version to prod.
4. Push every version and restore each into an empty folder.

- The exercise takes about 15 to 20 minutes.
- Only reading? The recorded output shows the starter. Follow the commands to complete it.

## The rule

- The scripts, parameters, pipeline lock and registry tags are pets. Keep them in Git.
- The model file is cattle. DVC stores it in a bucket so a new machine can restore it.
- GTO records which versions exist and which one is live. It needs no registry server.
- A registry works the same way for a large GPU-trained model. DVC stores the weights, and GTO tracks which version is live.

## Get a terminal

- You read this page and paste commands into a terminal. Nothing runs on the page.
- The output shown here is recorded from the untouched project, so it stops at the failures. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-04-registry?urlpath=lab/workspaces/exercise)**

- Free, with nothing to install. The first start can take a few minutes.
- The terminal is on the left, already in the `project` folder. This page is on the right.
- Copy a command: select it and press Ctrl+C, or Cmd+C on a Mac.
- Paste it in the terminal: Ctrl+Shift+V, or right-click and choose Paste. Cmd+V on a Mac is untested, so right-click if it doesn't work.
- Edit a file: double-click it in the file panel on the far left. Save with Ctrl+S.
- A session is temporary and has 2 GB of memory. Download anything you want to keep.

**Or use your own terminal**

- Optional: [use uv on your own machine](../../uv.md#exercise-4-model-registry).

- Linux or macOS, on x86_64 or ARM, or Windows through WSL2. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`.
- You need [conda](https://github.com/conda-forge/miniforge). You don't need Jupyter.
- Paste this once:

```bash
git clone --branch binder-04-registry --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git registry
cd registry
conda env create --quiet --file .binder/environment.yml
conda activate repro-registry
python scripts/setup_project.py
cd project
```

- Every command below runs in that `project` folder.
- To start over, go to the folder above it and run `python scripts/setup_project.py --reset`. This deletes your work.

## The project

- `dvc repro` runs five stages: clean, validate, report, split and train.
- The data is the published 2007 to 2009 table. Cleaning leaves 333 rows with recorded sex and body mass.
- `metrics.json` keeps the sexual dimorphism result, male mean mass minus female mean mass per species.
- `split.py` reads `results/validated.csv`. `splits.json` records 216 training row positions from 2007 and 2008 and 117 validation row positions from 2009.
- `train.py` standardizes the four measurements and one-hot species columns, then fits logistic regression.
- `params.yaml` sets `train.C`, the inverse regularization strength. `model_metrics.json` records `val_accuracy`.
- `artifacts.yaml` names the model `penguin-sex-clf` and points to `model.pkl`.
- `../osn-bucket` stands in for object storage such as OSN. Setup has pushed the first model there.

```bash
cat params.yaml
cat model_metrics.json
gto show
```

In [1]:
# Recorded output. You do not need to run this.
import sys
sys.path.insert(0, ".exercise")
from page import run
run("cat params.yaml", "cat model_metrics.json", "gto show")


$ cat params.yaml
report:
  statistic: mean          # or median
train:
  C: 0.01

$ cat model_metrics.json
{
  "val_accuracy": 0.846
}

$ gto show
╒══════════════════╤══════════╕
│ name             │ latest   │
╞══════════════════╪══════════╡
│ *penguin-sex-clf │ -        │
╘══════════════════╧══════════╛


## Step 1: register the first version

**Problem:** the first model exists, but the registry has no versions.

**Facts**

- `C: 0.01` gives validation accuracy 0.846, about 85 percent.
- `gto register` names a committed version with a Git tag.
- Setup has already committed the first model's parameters, pipeline lock and metrics.

**Steps**

1. Register the current commit as v1.0.0 and inspect the registry.

```bash
gto register penguin-sex-clf --version v1.0.0
gto show
```

## Step 2: train and register another version

**Problem:** one score does not tell you which value of C to use.

**Facts**

- Larger C means weaker regularization.
- The training and validation years stay the same for both versions.
- DVC reruns train when `train.C` changes. The analysis result in `metrics.json` stays the same.
- Commit before registering, so the version tag points to the parameters and metrics you actually used.

**Steps**

1. Open `params.yaml`, change `train.C` from `0.01` to `1`, and save it.
2. Train and read the validation score. It should be 0.889.

```bash
dvc repro
cat model_metrics.json
```

3. Commit and register v1.1.0.

```bash
git add params.yaml dvc.lock model_metrics.json
git commit --quiet -m "Train the penguin sex classifier with C=1"
gto register penguin-sex-clf --version v1.1.0
```

## Step 3: promote the better validation score

**Problem:** both versions are registered, but neither is marked as live.

**Facts**

- Compare `val_accuracy` in `model_metrics.json`.
- v1.1.0 scores 0.889, above v1.0.0 at 0.846.
- `gto assign` records a stage in a Git tag. Here, `prod` means the version chosen to be live.
- Assigning prod records a choice. It does not deploy a service.

**Steps**

1. Compare the current result with the first registered version.

```bash
dvc metrics diff penguin-sex-clf@v1.0.0 --targets model_metrics.json
```

2. Promote v1.1.0 and inspect the record.

```bash
gto assign penguin-sex-clf --version v1.1.0 --stage prod
gto show
gto history penguin-sex-clf
```

## Step 4: push and restore every version

**Problem:** a version tag does not contain the model's bytes.

**Facts**

- `dvc push` sends the current version's files to the bucket.
- `dvc push --all-tags` also sends the files referenced by older tags.
- Setup already pushed v1.0.0. Push all tags whenever you release, so older versions remain available.
- `restore.py` clones each registered tag into a separate empty folder and pulls from the bucket. It has none of your local DVC cache.
- A Git hosting service also needs your commits and tags. This local exercise uses no Git remote.

**Steps**

1. Push the current version, then every tagged version.

```bash
dvc push
dvc push --all-tags
```

2. Restore both registered models into empty folders.

```bash
python ../scripts/restore.py
```

- Both `penguin-sex-clf@v1.0.0` and `penguin-sex-clf@v1.1.0` should say `restored into an empty folder`.

## Check your work

- The check reads committed parameters and metrics at the registered tags.
- It requires two versions with different C values and prod at the highest validation accuracy.
- It restores prod into a clean clone and runs `dvc repro --force` to compare its validation accuracy with the record.
- It restores every registered version into a separate empty folder.
- `--short` runs this same check.

```bash
python ../check.py
```

On a fresh starter, before any versions are registered, it fails like this:

In [2]:
# Recorded output. You do not need to run this.
run("python ../check.py")


$ python ../check.py
FAIL  Register v1.0.0, train and commit a different C, then register v1.1.0 (steps 1 and 2).
FAIL  Compare model_metrics.json, then use gto assign penguin-sex-clf --version <best> --stage prod (step 3).
FAIL  Assign a registered version to prod before checking its clean rebuild.
FAIL  Register the versions and run dvc push --all-tags. Missing: no registered versions

0 of 4 parts pass.


## Summary

- GTO names committed model versions and records which version is live.
- Compare validation scores before assigning prod.
- DVC stores the model files. Push every tagged version and check restoration from an empty folder.

## On your own project

- Register the last model you shared, including its committed parameters and metrics.
- Choose a version using your project's validation measure.
- Restore that version on a clean machine before you depend on it.

## About the data

- Palmer penguins, CC0, 344 rows collected in 2007 to 2009. The exercise uses the published table unchanged.
- Data: palmerpenguins (Horst, Hill and Gorman 2020, doi:10.5281/zenodo.3960218), CC0, from Gorman, Williams and Fraser (2014), PLOS ONE 9(3): e90081. Collected by Palmer Station LTER with NSF support.